# Hough Dönüşümü ile Çizgi ve Daire Tespiti

Bu modül; bilgisayarlı görüde kısmi bozulmalara ve gürültüye rağmen parametrik geometrik şekilleri (doğrular ve çemberler) tespit etmek için kullanılan **Hough Dönüşümü** (Hough Transform) tekniğini inceler.

---

## 1. Matematiksel Temel

### Doğru Hough Dönüşümü
Kartezyen uzaydaki $y = mx + b$ doğrusu, dikey doğrular için $m \rightarrow \infty$ tanımsızlığına yol açar. Bu nedenle polar (kutupsal) koordinat sistemi kullanılır:
$$\rho = x \cos \theta + y \sin \theta$$
- $(x, y)$ görüntü uzayındaki her bir kenar noktası, $(\rho, \theta)$ parametre uzayında (akümülatör hücresi) bir sinüzoid eğri oluşturur.
- Aynı doğru üzerinde bulunan tüm noktaların eğrileri akümülatör üzerinde tek bir ortak tepe noktasında kesişir.

### Daire Hough Dönüşümü (Hough Gradient)
Bir çemberin denklemi 3 serbestlik derecesine sahiptir: $(x - a)^2 + (y - b)^2 = r^2$.
3 boyutlu bir akümülatör matrisi ($a, b, r$) hesaplama açısından çok maliyetli olacağından OpenCV **Hough Gradyan Yöntemi**'ni kullanır:
Kenar piksellerinin gradyan yönündeki doğrular merkez $a, b$ üzerinde kesişir, ardından yarıçap $r$ tespit edilir.


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Test görselini yükleme
image = cv2.imread('lanes_and_circles.png')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Kenar Tespiti
edges = cv2.Canny(gray, 50, 150, apertureSize=3)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.title("Orijinal Test Görseli")
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Canny Kenar Haritası")
plt.imshow(edges, cmap='gray')
plt.axis('off')
plt.show()


## 2. Olasılıksal Hough Doğru Tespiti (`cv2.HoughLinesP`)

`cv2.HoughLinesP(image, rho, theta, threshold, minLineLength, maxLineGap)`:
- `rho`: Mesafe çözünürlüğü (genellikle 1 piksel).
- `theta`: Açı çözünürlüğü (genellikle 1 derece: $\pi / 180$).
- `threshold`: Bir doğrunun kabul edilmesi için gereken minimum oy sayısı.
- `minLineLength`: Kabul edilecek en kısa doğru uzunluğu.
- `maxLineGap`: Aynı doğru üzerindeki kırık çizgileri birleştirmek için izin verilen maksimum boşluk.


In [ ]:
lines_image = image.copy()

# Olasılıksal Hough ile çizgileri bulma
lines = cv2.HoughLinesP(edges, rho=1, theta=np.pi/180, threshold=40,
                        minLineLength=50, maxLineGap=20)

if lines is not None:
    print(f"Toplam Tespit Edilen Çizgi Segmenti: {len(lines)}")
    for line in lines:
        x1, y1, x2, y2 = line[0]
        cv2.line(lines_image, (x1, y1), (x2, y2), (0, 255, 0), 3)

plt.figure(figsize=(8, 6))
plt.title("Hough Doğru Tespiti (Yeşil Çizgiler)")
plt.imshow(cv2.cvtColor(lines_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()


## 3. Hough Daire Tespiti (`cv2.HoughCircles`)

`cv2.HoughCircles(image, method, dp, minDist, param1, param2, minRadius, maxRadius)`:
- `dp`: Akümülatör çözünürlüğünün giriş görüntüsüne oranı ($1.0$ birebir çözünürlük).
- `minDist`: Tespit edilen çember merkezleri arasındaki minimum mesafe.
- `param1`: Canny kenar dedektörünün üst eşiği.
- `param2`: Merkez akümülatörünün tepe oyu eşiği (Düşürülürse daha fazla çember bulunur, artırılırsa hassasiyet yükselir).


In [ ]:
circles_image = image.copy()

# Gürültü azaltma için hafif Gauss yumuşatma
blurred = cv2.GaussianBlur(gray, (9, 9), 2)

circles = cv2.HoughCircles(blurred, cv2.HOUGH_GRADIENT, dp=1.2, minDist=50,
                           param1=100, param2=30, minRadius=15, maxRadius=100)

if circles is not None:
    circles = np.uint16(np.around(circles))
    print(f"Tespit Edilen Çember Sayısı: {len(circles[0, :])}")
    for c in circles[0, :]:
        center = (c[0], c[1])
        radius = c[2]
        # Dış çember
        cv2.circle(circles_image, center, radius, (255, 0, 0), 3)
        # Merkez noktası
        cv2.circle(circles_image, center, 3, (0, 0, 255), -1)

plt.figure(figsize=(8, 6))
plt.title("Hough Daire Tespiti (Mavi Çemberler)")
plt.imshow(cv2.cvtColor(circles_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()
